In [2]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Configuración de rutas (asegúrate de que ROOT suba a la raíz)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]

# comentario 

In [3]:
in_kind_transfers = pd.read_csv(PROCESSED_DIR / "in_kind_transfers.csv",
                        dtype={'folioviv': str, 'foliohog': str, 'numren': str})


iva_hogar = pd.read_csv(PROCESSED_DIR / "iva_hogar.csv",
                        dtype={'folioviv': str, 'foliohog': str})


income_fiscal_incidence_base = pd.read_csv(PROCESSED_DIR / "income_fiscal_incidence_base.csv",
                        dtype={'folioviv': str, 'foliohog': str, 'numren':str})


demographic_household = pd.read_csv(PROCESSED_DIR / "demographic_household.csv",
                        dtype={'folioviv': str, 'foliohog': str})




/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_75403/2259054953.py:1: DtypeWarning: Columns (0: etnia) have mixed types. Specify dtype option on import or set low_memory=False.
  in_kind_transfers = pd.read_csv(PROCESSED_DIR / "in_kind_transfers.csv",


In [21]:
import pandas as pd

# ==========================================================
# 1. Número de integrantes por hogar
# ==========================================================

household_size = (
    income_fiscal_incidence_base
    .groupby(["folioviv", "foliohog", "year"])
    .size()
    .reset_index(name="household_size")
)

# ==========================================================
# 2. IVA per cápita
# ==========================================================

iva_hogar = iva_hogar.merge(
    household_size,
    on=["folioviv", "foliohog", "year"],
    how="inner"
)

iva_cols = [
    "gasto_con_iva",
    "gasto_sin_iva",
    "iva_pagado",
    "gasto_total",
    "gasto_gas",
    "gasto_electricity"
]

iva_hogar[iva_cols] = iva_hogar[iva_cols].div(
    iva_hogar["household_size"],
    axis=0
)

# Si ya no la necesitas
iva_hogar = iva_hogar.drop(columns="household_size")

# ==========================================================
# 3. Base a nivel hogar
# ==========================================================

household_base = (
    demographic_household
    .merge(
        iva_hogar,
        on=["folioviv", "foliohog", "year"],
        how="inner"
    )
)

# ==========================================================
# 4. Llevar variables del hogar a la base individual
# ==========================================================

individual_base = (
    income_fiscal_incidence_base
    .merge(
        household_base,
        on=["folioviv", "foliohog", "year"],
        how="inner"
    )
)

# ==========================================================
# 5. Agregar transferencias en especie
# ==========================================================

# Asegurar que las llaves tengan el mismo tipo
individual_base["folioviv"] = individual_base["folioviv"].astype(str)
individual_base["foliohog"] = individual_base["foliohog"].astype(str)
individual_base["numren"] = individual_base["numren"].astype(str)

in_kind_transfers["folioviv"] = in_kind_transfers["folioviv"].astype(str)
in_kind_transfers["foliohog"] = in_kind_transfers["foliohog"].astype(str)
in_kind_transfers["numren"] = in_kind_transfers["numren"].astype(str)

# Merge
individual_base = individual_base.merge(
    in_kind_transfers,
    on=["folioviv", "foliohog", "numren", "year"],
    how="inner"
)


# ==========================================================
# Resultado
# ==========================================================

print(individual_base.head())

individual_base.to_csv(PROCESSED_DIR / "base_final.csv", index=False)
print("Saved base_final.csv")

     folioviv foliohog numren  year  net_trabajo1_ann  net_trabajo2_ann  \
0  1000021401        1      2  2018         322826.04              0.00   
1  1000021401        1      3  2018              0.00          89021.68   
2  1000021401        1      1  2018              0.00              0.00   
3  1000021402        1      1  2018              0.00              0.00   
4  1000021402        1      4  2018              0.00              0.00   

      isr_total  formal_trabajo_1  formal_trabajo_2   mi_trabajo_1  ...  \
0  62521.198849               1.0               0.0  385347.238849  ...   
1   6598.270485               0.0               1.0       0.000000  ...   
2      0.000000               0.0               0.0       0.000000  ...   
3      0.000000               0.0               0.0       0.000000  ...   
4      0.000000               0.0               0.0       0.000000  ...   

   gasto_sin_iva  iva_pagado  gasto_total   gasto_gas  gasto_electricity  \
0      20.916420   54.

In [23]:
print(f"Total de registros: {len(individual_base)}")
print(f"Registros para 2018: {len(individual_base[individual_base['year'] == 2018])}")
print(f"Registros para 2024: {len(individual_base[individual_base['year'] == 2024])}")

Total de registros: 267486
Registros para 2018: 127622
Registros para 2024: 139864


In [24]:
import pandas as pd

# ==========================================================
# 1. Número de integrantes por hogar
# ==========================================================

print("\n--- Paso 1: household_size ---")
print(f"Tamaño de income_fiscal_incidence_base (antes): {len(income_fiscal_incidence_base)}")

household_size = (
    income_fiscal_incidence_base
    .groupby(["folioviv", "foliohog", "year"])
    .size()
    .reset_index(name="household_size")
)

print(f"Tamaño de household_size (después): {len(household_size)}")

# ==========================================================
# 2. IVA per cápita
# ==========================================================

print("\n--- Paso 2: IVA per cápita ---")
print(f"Tamaño de iva_hogar (antes del merge con household_size): {len(iva_hogar)}")

iva_hogar = iva_hogar.merge(
    household_size,
    on=["folioviv", "foliohog", "year"],
    how="inner"
)

print(f"Tamaño de iva_hogar (después del merge): {len(iva_hogar)}")

iva_cols = [
    "gasto_con_iva",
    "gasto_sin_iva",
    "iva_pagado",
    "gasto_total",
    "gasto_gas",
    "gasto_electricity"
]

iva_hogar[iva_cols] = iva_hogar[iva_cols].div(
    iva_hogar["household_size"],
    axis=0
)

# Si ya no la necesitas
iva_hogar = iva_hogar.drop(columns="household_size")

# ==========================================================
# 3. Base a nivel hogar
# ==========================================================

print("\n--- Paso 3: Base a nivel hogar ---")
print(f"Tamaño de demographic_household (antes del merge): {len(demographic_household)}")
print(f"Tamaño de iva_hogar (antes del merge): {len(iva_hogar)}")

household_base = (
    demographic_household
    .merge(
        iva_hogar,
        on=["folioviv", "foliohog", "year"],
        how="inner"
    )
)

print(f"Tamaño de household_base (después del merge): {len(household_base)}")

# ==========================================================
# 4. Llevar variables del hogar a la base individual
# ==========================================================

print("\n--- Paso 4: Base individual con variables del hogar ---")
print(f"Tamaño de income_fiscal_incidence_base (antes del merge): {len(income_fiscal_incidence_base)}")
print(f"Tamaño de household_base (antes del merge): {len(household_base)}")

individual_base = (
    income_fiscal_incidence_base
    .merge(
        household_base,
        on=["folioviv", "foliohog", "year"],
        how="inner"
    )
)

print(f"Tamaño de individual_base (después del merge): {len(individual_base)}")

# ==========================================================
# 5. Agregar transferencias en especie
# ==========================================================

print("\n--- Paso 5: Agregar transferencias en especie ---")

# Asegurar que las llaves tengan el mismo tipo
individual_base["folioviv"] = individual_base["folioviv"].astype(str)
individual_base["foliohog"] = individual_base["foliohog"].astype(str)
individual_base["numren"] = individual_base["numren"].astype(str)

in_kind_transfers["folioviv"] = in_kind_transfers["folioviv"].astype(str)
in_kind_transfers["foliohog"] = in_kind_transfers["foliohog"].astype(str)
in_kind_transfers["numren"] = in_kind_transfers["numren"].astype(str)

print(f"Tamaño de individual_base (antes del merge con transfers): {len(individual_base)}")
print(f"Tamaño de in_kind_transfers (antes del merge): {len(in_kind_transfers)}")

# Merge
individual_base = individual_base.merge(
    in_kind_transfers,
    on=["folioviv", "foliohog", "numren", "year"],
    how="inner"
)

print(f"Tamaño de individual_base (después del merge con transfers): {len(individual_base)}")

# ==========================================================
# 6. Conteo por año (2018 y 2024)
# ==========================================================

print("\n--- Conteo de registros por año en la base final ---")
print(individual_base['year'].value_counts().sort_index())

# Si quieres ver específicamente 2018 y 2024:
print(f"\nRegistros para 2018: {len(individual_base[individual_base['year'] == 2018])}")
print(f"Registros para 2024: {len(individual_base[individual_base['year'] == 2024])}")

# ==========================================================
# Resultado
# ==========================================================

print("\n--- Muestra de la base final ---")
print(individual_base.head())

individual_base.to_csv(PROCESSED_DIR / "base_final.csv", index=False)
print("Saved base_final.csv")


--- Paso 1: household_size ---
Tamaño de income_fiscal_incidence_base (antes): 384799
Tamaño de household_size (después): 165949

--- Paso 2: IVA per cápita ---
Tamaño de iva_hogar (antes del merge con household_size): 165949
Tamaño de iva_hogar (después del merge): 165949

--- Paso 3: Base a nivel hogar ---
Tamaño de demographic_household (antes del merge): 166061
Tamaño de iva_hogar (antes del merge): 165949
Tamaño de household_base (después del merge): 113454

--- Paso 4: Base individual con variables del hogar ---
Tamaño de income_fiscal_incidence_base (antes del merge): 384799
Tamaño de household_base (antes del merge): 113454
Tamaño de individual_base (después del merge): 267486

--- Paso 5: Agregar transferencias en especie ---
Tamaño de individual_base (antes del merge con transfers): 267486
Tamaño de in_kind_transfers (antes del merge): 577804
Tamaño de individual_base (después del merge con transfers): 267486

--- Conteo de registros por año en la base final ---
year
2018   

In [25]:
demographic_household["year"].value_counts().sort_index()

iva_hogar["year"].value_counts().sort_index()

year
2018    74597
2024    91352
Name: count, dtype: int64

In [26]:
demographic_household.groupby("year").size()

iva_hogar.groupby("year").size()

year
2018    74597
2024    91352
dtype: int64

In [27]:
for y in [2018, 2024]:
    print(f"\nAño {y}")

    d = demographic_household.query("year == @y")
    i = iva_hogar.query("year == @y")

    print(len(d))
    print(len(i))

    t = d.merge(
        i,
        on=["folioviv","foliohog","year"],
        how="outer",
        indicator=True
    )

    print(t["_merge"].value_counts())


Año 2018
74647
74597
_merge
both          51090
left_only     23557
right_only    23507
Name: count, dtype: int64

Año 2024
91414
91352
_merge
both          62364
left_only     29050
right_only    28988
Name: count, dtype: int64


In [28]:
demographic_household["foliohog"].value_counts().head()

iva_hogar["foliohog"].value_counts().head()

foliohog
1    163617
2      2022
3       259
4        46
5         5
Name: count, dtype: int64

In [29]:
print(demographic_household["foliohog"].unique()[:10])
print(iva_hogar["foliohog"].unique()[:10])

<StringArray>
['1', '2', '3', '4', '5']
Length: 5, dtype: str
<StringArray>
['1', '2', '3', '4', '5']
Length: 5, dtype: str
